<a href="https://colab.research.google.com/github/beingAnujChaudhary/Customer-Churn-Prediction-Retention-Prioritization/blob/main/notebooks/03_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% [markdown]
# # 🎛️ Stage 6: Hyperparameter Tuning
# *Notebook: 03_tuning.ipynb*
#
# **Purpose**: Optimize XGBoost via RandomizedSearchCV (5-fold CV)
# **Target**: Beat baseline AUC (0.87) → Aim for ≥0.89

# %% [code]
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import joblib
import os
from datetime import datetime

# Setup directories
os.makedirs("output/models", exist_ok=True)
os.makedirs("output/plots", exist_ok=True)
os.makedirs("experiments", exist_ok=True)

print("✓ Environment ready")

✓ Environment ready


In [3]:
# %% [code]
# LOAD FEATURE-ENGINEERED DATA
df = pd.read_csv("customers_v3.csv")

train_df = df[df["split"] == "train"].drop(columns=["split"])
test_df = df[df["split"] == "test"].drop(columns=["split"])

X_train = train_df.drop(columns=["Churn"])
y_train = train_df["Churn"]
X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"   Baseline XGBoost AUC: 0.8734 (from Stage 5)")

✅ Train: 8101 | Test: 2026
   Baseline XGBoost AUC: 0.8734 (from Stage 5)


In [4]:
# %% [code]
# HYPERPARAMETER SEARCH SPACE
param_dist = {
    'n_estimators': [100, 150, 200, 250],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [len(y_train[y_train==0]) / len(y_train[y_train==1])],  # Fixed imbalance ratio
    'eval_metric': ['auc'],
    'random_state': [42]
}

print("🔍 Search space defined:")
for k, v in param_dist.items():
    print(f"   {k}: {v}")

🔍 Search space defined:
   n_estimators: [100, 150, 200, 250]
   max_depth: [3, 5, 7, 9]
   learning_rate: [0.01, 0.05, 0.1, 0.15]
   subsample: [0.7, 0.8, 0.9, 1.0]
   colsample_bytree: [0.7, 0.8, 0.9, 1.0]
   scale_pos_weight: [5.221966205837173]
   eval_metric: ['auc']
   random_state: [42]


In [5]:
# %% [code]
# RUN RANDOMIZED SEARCH (5-FOLD CV)
print("🚀 Starting RandomizedSearchCV (20 iterations, 5-fold CV)...")

xgb_base = xgb.XGBClassifier(use_label_encoder=False, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print(f"\n✅ Tuning complete!")
print(f"🏆 Best AUC (CV): {search.best_score_:.4f}")
print(f"⚙️ Best Parameters:")
for k, v in search.best_params_.items():
    print(f"   {k}: {v}")

🚀 Starting RandomizedSearchCV (20 iterations, 5-fold CV)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:50:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ Tuning complete!
🏆 Best AUC (CV): 0.9933
⚙️ Best Parameters:
   subsample: 0.7
   scale_pos_weight: 5.221966205837173
   random_state: 42
   n_estimators: 250
   max_depth: 9
   learning_rate: 0.1
   eval_metric: auc
   colsample_bytree: 0.8


In [6]:
# %% [code]
# EVALUATE ON TEST SET
best_model = search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]

tuned_auc = roc_auc_score(y_test, y_proba_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

# Baseline metrics (from Stage 5)
baseline_auc = 0.8734
baseline_f1 = 0.7640

print("\n📊 PERFORMANCE COMPARISON")
print("="*60)
print(f"Metric        | Baseline XGBoost | Tuned XGBoost | Δ Change")
print("-"*60)
print(f"AUC           | {baseline_auc:.4f}           | {tuned_auc:.4f}       | {tuned_auc-baseline_auc:+.4f}")
print(f"F1-Score      | {baseline_f1:.4f}           | {tuned_f1:.4f}       | {tuned_f1-baseline_f1:+.4f}")
print("="*60)

if tuned_auc > baseline_auc:
    print("🎉 Tuning successful! Model improved.")
else:
    print("⚠️ Tuning didn't beat baseline. Keeping tuned params for robustness.")


📊 PERFORMANCE COMPARISON
Metric        | Baseline XGBoost | Tuned XGBoost | Δ Change
------------------------------------------------------------
AUC           | 0.8734           | 0.9911       | +0.1177
F1-Score      | 0.7640           | 0.8968       | +0.1328
🎉 Tuning successful! Model improved.


In [7]:
# %% [code]
# VISUALIZE IMPROVEMENT
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
models = ['Baseline\nXGBoost', 'Tuned\nXGBoost']
auc_scores = [baseline_auc, tuned_auc]
f1_scores = [baseline_f1, tuned_f1]

x = np.arange(len(models))
width = 0.35

ax.bar(x - width/2, auc_scores, width, label='AUC', color='#4CAF50')
ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#2196F3')

ax.set_ylabel('Score')
ax.set_title('XGBoost: Baseline vs Tuned Performance', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim([0.7, 1.0])

# Add value labels
for i, v in enumerate(auc_scores):
    ax.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
for i, v in enumerate(f1_scores):
    ax.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("output/plots/tuning_comparison.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: output/plots/tuning_comparison.png")

✅ Saved: output/plots/tuning_comparison.png


In [9]:
# %% [code]
# SAVE TUNED MODEL
joblib.dump(best_model, "output/models/xgboost_tuned.pkl")
print("✅ Saved: output/models/xgboost_tuned.pkl")

# LOG EXPERIMENT
exp_log = f"""# Experiment 02: XGBoost Hyperparameter Tuning

## Metadata
- **Date**: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- **Purpose**: Optimize XGBoost via RandomizedSearchCV
- **Status**: ✅ Complete

## Results
| Metric | Baseline | Tuned | Δ |
|--------|----------|-------|---|
| AUC | {baseline_auc:.4f} | {tuned_auc:.4f} | {tuned_auc-baseline_auc:+.4f} |
| F1-Score | {baseline_f1:.4f} | {tuned_f1:.4f} | {tuned_f1-baseline_f1:+.4f} |

## Best Parameters
```python
{search.best_params_}
```

## CV Performance
- Best CV AUC: {search.best_score_:.4f}
- CV Folds: 5 (Stratified)
- Iterations: 20

## Artifacts
- Model: `output/models/xgboost_tuned.pkl`
- Plot: `output/plots/tuning_comparison.png`
- Data: `data/processed/customers_v3.csv`

## Notes
- Used `scale_pos_weight` to handle class imbalance
- Search space aligned with README methodology
- Next: SHAP explainability on tuned model
"""

# Write to file
with open("experiments/experiment_02.md", "w", encoding="utf-8") as f:
    f.write(exp_log)

print("✅ Logged: experiments/experiment_02.md")


✅ Saved: output/models/xgboost_tuned.pkl
✅ Logged: experiments/experiment_02.md
